In [1]:
import os
# —— 1. 环境变量 & 路径 ——
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from unsloth import FastLanguageModel
import json, torch
from datasets import load_dataset, concatenate_datasets
from transformers import Trainer, TrainingArguments

MODEL_DIR = "" # where to save
DATA_PATH = "" # where to load training data
MAX_LEN = 16384

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/yang3j7/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# —— 2. 加载原始数据集 ——
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# —— 3. 加载模型 & tokenizer ——
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-14B",  # different base model to finetune
    max_seq_length = MAX_LEN,                           # input length
    dtype          = torch.bfloat16,
    load_in_4bit   = False,
)
model.train()


==((====))==  Unsloth 2025.8.5: Fast Qwen3 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 6/6 [00:07<00:00,  1.21s/it]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 5120, padding_idx=151654)
    (layers): ModuleList(
      (0-39): 40 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=False)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (up_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (down_proj): Linear(in_features=17408, out_features=5120, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qw

In [4]:
# —— 4. 定义 system + prompt template ——  can be customized
system_msg = {
    "role": "system",
    "content": "You are a helpful assistant."
}

prompt_tpl = """### Instruction:
Please determine whether there is a non-crash functional bug in the following android app ui interaction trace, and briefly explain the reason.

### Input:
{actions}

### Output format:
{{
  "is_bug": "Yes" or "No",
  "reason": "brief reasons for why consider as bug or no bug."
}}
"""


In [5]:
def _norm_is_bug(x):
    # 统一 "Yes"/"No"
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"yes", "true", "1"}:
            return "Yes"
        if s in {"no", "false", "0"}:
            return "No"
        # 兜底：未知字符串按 No 处理（也可改成抛错）
        return "No"
    if isinstance(x, (bool, int)):
        return "Yes" if bool(x) else "No"
    return "No"

def format_example(example):
    # 1) 读取新结构
    trace = example.get("gen_trace", "")
    j = example.get("judgement", {}) or {}

    # 2) 规范化标签
    is_bug = _norm_is_bug(j.get("is_bug", "No"))
    reason = (j.get("reason") or "").strip()

    # 3) 组 prompt（system + user 到“准备生成”为止）
    user_msg = {
        "role": "user",
        "content": prompt_tpl.format(actions=trace),
    }
    prompt_text = tokenizer.apply_chat_template(
        [system_msg, user_msg],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # 4) 目标输出（加 eos）
    out_obj = {"is_bug": is_bug, "reason": reason}
    output_text = json.dumps(out_obj, ensure_ascii=False) + tokenizer.eos_token

    # 5) 拼接 & tokenize（显式 max_length=8192 防意外）
    full_text = prompt_text + output_text
    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding="longest",
    )
    prompt_tokens = tokenizer(prompt_text, add_special_tokens=False)

    input_ids = full_tokens["input_ids"]
    prompt_len = len(prompt_tokens["input_ids"])

    # 6) 只让 output 段参与 loss
    labels = [-100] * prompt_len + input_ids[prompt_len:]

    # 7) 保护：若异常则跳过该样本
    if len(labels) != len(input_ids) or all(l == -100 for l in labels):
        return {}

    full_tokens["labels"] = labels
    return full_tokens

# 5.7 把处理函数 map 到 dataset
dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
)

Map: 100%|██████████| 3546/3546 [00:18<00:00, 188.02 examples/s]


In [6]:
# —— 6. 应用 LoRA（peft） —— 
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha         = 32,
    lora_dropout       = 0.05,
    bias               = "none",
    use_gradient_checkpointing = True,
    random_state       = 42,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.8.5 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [7]:
# —— 7. TrainingArguments：batch=1 —— 
# MODEL_DIR = "/home/yang3j7/finetune/test0717/new_trained_model_prompt_with_chat_template_with_output_format"

training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
)

In [8]:
# —— 8. 初始化 Trainer ——
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset,
    processing_class = tokenizer,
)

In [9]:
# 9. 开始训练 and save 
trainer.train()
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,546 | Num Epochs = 3 | Total steps = 10,638
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 20,971,520 of 14,789,278,720 (0.14% trained)


Step,Training Loss
20,4.405600
40,4.079600
60,3.314300
80,2.146600
100,1.944900
120,1.606300
140,1.507800
160,1.301800
180,1.329800
200,1.219000


Unsloth: Will smartly offload gradients to save VRAM!


('/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/tokenizer_config.json',
 '/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/special_tokens_map.json',
 '/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/chat_template.jinja',
 '/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/vocab.json',
 '/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/merges.txt',
 '/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/added_tokens.json',
 '/home/yang3j7/NCF0729/test0923/sft_qwen3_new_sorted_batch2/tokenizer.json')